In [5]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# --- CONFIGURATION ---
input_csv_name = None
for f in os.listdir('.'): 
    if f.endswith('_processed.csv'):
        input_csv_name = f
        break

if not input_csv_name:
    print("ERRO: Nenhum arquivo '*_processed.csv' encontrado.")
else:
    print(f"Processando arquivo: {input_csv_name}")

    # Lista FINAL de colunas desejadas (UNSW-NB15)
    target_columns = [
        'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
        'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
        'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean',
        'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
        'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login',
        'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports',
        'attack_cat', 'label'
    ]

    rename_map = {
        'duration': 'dur',
        'conn_state': 'state',
        'orig_pkts': 'spkts',
        'resp_pkts': 'dpkts',
        'orig_bytes': 'sbytes',
        'resp_bytes': 'dbytes',
        'http_trans_depth': 'trans_depth',
        'http_response_body_len': 'response_body_len'
        # sttl, dttl, sjit, etc já vêm com nomes certos do data-processing
    }

    # Valores padrão para preenchimento se coluna não existir
    default_values = {
        'sttl': 0, 'dttl': 0, 'sloss': 0, 'dloss': 0,
        'sinpkt': 0.0, 'dinpkt': 0.0, 'sjit': 0.0, 'djit': 0.0,
        'swin': 0, 'stcpb': 0, 'dtcpb': 0, 'dwin': 0,
        'tcprtt': 0.0, 'synack': 0.0, 'ackdat': 0.0,
        'trans_depth': 0, 'response_body_len': 0, 'is_ftp_login': 0
    }

    try:
        df = pd.read_csv(input_csv_name, low_memory=False)
        print(f"Lido: {len(df)} linhas.")

        df.rename(columns=rename_map, inplace=True)

        # Adiciona colunas faltantes com default
        for col in target_columns:
            if col not in df.columns:
                df[col] = default_values.get(col, 0)
        
        # --- ENGENHARIA DE FEATURES AGREGADAS (ct_*) ---
        print("Calculando features de janela (ct_*)...")
        df_sorted = df.sort_values(by='ts').reset_index(drop=True) if 'ts' in df.columns else df.copy()
        window_size = 100
        
        # (Lógica simplificada de loop para IPs omitida aqui por brevidade, assumindo que você usa o código já existente no seu script para ct_srv_src, etc.)
        # ... Insira seu loop de ct_srv_src, ct_dst_ltm aqui se necessário ...
        
        # --- FEATURE COMPLEXA: ct_state_ttl ---
        print("Calculando ct_state_ttl...")
        
        if 'sttl' in df_sorted.columns and 'dttl' in df_sorted.columns and 'state' in df_sorted.columns:
             # Verifica se STTL tem dados reais (Argus)
             if df_sorted['sttl'].sum() > 0:
                 print("  Usando dados REAIS do Argus (State + STTL + DTTL).")
                 df_sorted['sttl'] = df_sorted['sttl'].fillna(0).astype(int)
                 df_sorted['dttl'] = df_sorted['dttl'].fillna(0).astype(int)
                 # Lógica UNSW: Contagem da combinação
                 df_sorted['ct_state_ttl'] = df_sorted.groupby(['state', 'sttl', 'dttl'])['state'].transform('count')
             else:
                 print("  STTL zerado. Usando Fallback (apenas State).")
                 df_sorted['ct_state_ttl'] = df_sorted.groupby('state')['state'].transform('count')
        elif 'state' in df_sorted.columns:
             df_sorted['ct_state_ttl'] = df_sorted.groupby('state')['state'].transform('count')
        else:
             df_sorted['ct_state_ttl'] = 0

        # Seleção Final
        df_final = df_sorted[target_columns]
        
        output_csv_name = input_csv_name.replace('_processed.csv', '_features_processed.csv')
        df_final.to_csv(output_csv_name, index=False)
        print(f"Salvo: {output_csv_name}")

    except Exception as e:
        print(f"Erro: {e}")

Processando arquivo: normal_processed.csv
Lido: 410 linhas.
Calculando features de janela (ct_*)...
Calculando ct_state_ttl...
  Usando dados REAIS do Argus (State + STTL + DTTL).
Salvo: normal_features_processed.csv
